# Setup

In [1]:
from shapely.geometry import box
import geopandas as gpd
import geopandas as gpd
import leafmap
import pandas as pd
import geoai

In [2]:
# the Syke peat after-use layer
peat = gpd.read_file("../data/raw/vectors/turvetuotantoalueet/turvetuotantoalueet_jalkikaytto.shp")

# leafmap needs WGS84 for display
peat_wgs = peat.to_crs("EPSG:4326")

# one colour per after-use class
colors = {
    "avoin": "#ff6600",
    "metsittynyt": "#00e64d",
    "turvetuotanto": "#cc00ff",
    "vesi": "#00ccff",
    "maatalous": "#ffee00",
    "rakennettu": "#ff0033",}

# same colours, English labels for the legend
legend = {
    "Open": "#ff6600",
    "Afforested": "#00e64d",
    "Still in production": "#cc00ff",
    "Water": "#00ccff",
    "Agriculture": "#ffee00",
    "Built-up": "#ff0033",}

print(f"{len(peat)} polygons loaded, CRS {peat.crs}")

164821 polygons loaded, CRS PROJCS["EUREF_FIN_TM35FIN",GEOGCS["EUREF-FIN",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",27],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [3]:
# English labels for the Finnish class names
class_names_en = {
    "avoin": "open",
    "metsittynyt": "afforested",
    "turvetuotanto": "still_in_production",
    "vesi": "water",
    "maatalous": "agriculture",
    "rakennettu": "built_up",
}

peat_wgs["class_en"] = peat_wgs["luokka"].map(class_names_en)


# Inspection of training data

In [4]:
# slide a 0.4 x 0.3 degree window over the region and count what falls inside
results = []

for west in [21.8, 22.0, 22.2, 22.4, 22.6, 22.8, 23.0, 23.2, 23.4, 23.6, 23.8]:
    for south in [62.0, 62.2, 62.4, 62.6, 62.8, 63.0]:
        east = west + 0.4
        north = south + 0.3
        window = peat_wgs.cx[west:east, south:north]
        counts = window["class_en"].value_counts()
        results.append({
            "west": west,
            "south": south,
            "still_in_production": counts.get("still_in_production", 0),
            "open": counts.get("open", 0),
            "total": len(window),
        })

candidates = pd.DataFrame(results)
print(candidates.sort_values("still_in_production", ascending=False).head(15))

    west  south  still_in_production  open  total
19  22.4   62.2                  193  1576   2399
43  23.2   62.2                  192  2349   3779
37  23.0   62.2                  176  2558   4229
2   21.8   62.4                  171   696   1155
25  22.6   62.2                  154  1990   2914
24  22.6   62.0                  149  2177   3549
26  22.6   62.4                  139  1548   2331
20  22.4   62.4                  138  1062   1579
44  23.2   62.4                  122  1955   3294
18  22.4   62.0                  119  1075   1893
12  22.2   62.0                  117  1120   1757
50  23.4   62.4                  113  1400   2116
32  22.8   62.4                  112  1440   2249
31  22.8   62.2                  112  1853   2857
13  22.2   62.2                  112   715   1198


In [5]:
# training region, from the window search, 193 production sites
train_area = {"west": 22.4, "south": 62.2, "east": 22.8, "north": 62.5}

# test region, 192 production sites
test_area = {"west": 23.2, "south": 62.2, "east": 23.6, "north": 62.5}

train_box = gpd.GeoDataFrame(
    {"name": ["train"]},
    geometry=[box(train_area["west"], train_area["south"], train_area["east"], train_area["north"])],
    crs="EPSG:4326",)

test_box = gpd.GeoDataFrame(
    {"name": ["test"]},
    geometry=[box(test_area["west"], test_area["south"], test_area["east"], test_area["north"])],
    crs="EPSG:4326",)

# train and test regions added to the legend
legend_with_boxes = dict(legend)
legend_with_boxes["Train region"] = "#ffffff"
legend_with_boxes["Test region"] = "#000000"

# only the polygons near the two boxes, drawing all of Finland would hang the browser
peat_near = peat_wgs.cx[22.0:24.0, 62.0:63.3]

m = leafmap.Map(center=[62.5, 22.9], zoom=9)

# The layer "rakkenettu" is left out as there is only one polygon in the whole dataset.
# one layer per class, each gets its own toggle in the layer control
m.add_gdf(peat_near[peat_near["luokka"] == "avoin"], layer_name="Open",
          style={"color": "#ff6600", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "metsittynyt"], layer_name="Afforested",
          style={"color": "#00e64d", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "turvetuotanto"], layer_name="Still in production",
          style={"color": "#cc00ff", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "vesi"], layer_name="Water",
          style={"color": "#00ccff", "weight": 2, "fill": False})
m.add_gdf(peat_near[peat_near["luokka"] == "maatalous"], layer_name="Agriculture",
          style={"color": "#ffee00", "weight": 2, "fill": False})

# the two regions on top
m.add_gdf(train_box, layer_name="Train region", style={"color": "#ffffff", "weight": 3, "fill": False})
m.add_gdf(test_box, layer_name="Test region", style={"color": "#000000", "weight": 3, "fill": False})

m.add_legend(title="After-use class", legend_dict=legend_with_boxes)
m

Map(center=[62.5, 22.9], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out…

In [6]:
train_window = peat_wgs.cx[22.4:22.8, 62.2:62.5]
test_window = peat_wgs.cx[23.2:23.6, 62.2:62.5]

print("TRAIN")
print(train_window["class_en"].value_counts())

print("TEST")
print(test_window["class_en"].value_counts())

TRAIN
class_en
open                   1576
afforested              388
still_in_production     193
agriculture             125
water                   117
Name: count, dtype: int64
TEST
class_en
open                   2349
afforested             1084
still_in_production     192
agriculture              80
water                    74
Name: count, dtype: int64


In [7]:
# Checking the raster data for the training region
geoai.print_raster_info("../data/raw/raster/binary_model_train_2021.tif", show_preview=False)

===== RASTER INFORMATION: ../data/raw/raster/binary_model_train_2021.tif =====
Driver: GTiff
Dimensions: 2155 x 3394 pixels
Number of bands: 6
Data type: int16
Coordinate Reference System: EPSG:32634
Georeferenced Bounds: BoundingBox(left=572110.0, bottom=6897240.0, right=593660.0, top=6931180.0)
Pixel Resolution: 10.0, 10.0
NoData Value: -32768.0
----- Band Statistics -----
Band 1:
  Min: 45.00
  Max: 12708.00
  Mean: 310.46
  Std Dev: 133.98
Band 2:
  Min: 64.00
  Max: 12756.00
  Mean: 480.11
  Std Dev: 181.66
Band 3:
  Min: 49.00
  Max: 10376.00
  Mean: 369.77
  Std Dev: 224.01
Band 4:
  Min: 38.00
  Max: 12124.00
  Mean: 2537.19
  Std Dev: 831.95
Band 5:
  Min: 25.00
  Max: 6211.00
  Mean: 1567.23
  Std Dev: 612.32
Band 6:
  Min: 14.00
  Max: 5990.00
  Mean: 859.62
  Std Dev: 486.57


In [8]:
peat_aoi = peat_wgs.cx[22.4:22.8, 62.2:62.5]

m = leafmap.Map(center=[62.35, 22.6], zoom=11)
m.add_raster("../data/raw/raster/binary_model_train_ndvi_2021.tif", colormap="RdYlGn", vmin=0, vmax=0.9, layer_name="NDVI")

for cls, c in colors.items():
    sub = peat_aoi[peat_aoi["luokka"] == cls]
    if len(sub) > 0:
        m.add_gdf(sub, layer_name=cls, style={"color": c, "weight": 2, "fill": False})

m.add_legend(title="After-use class", legend_dict=legend)
m

Map(center=[62.349972, 22.6020605], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

In [9]:
train_window = peat_wgs.cx[22.4:22.8, 62.2:62.5].to_crs(3067)
train_window["area_ha"] = train_window.area / 10000

summary = train_window.groupby("class_en")["area_ha"].sum().sort_values(ascending=False)
print("area in hectares:")
print(summary)


area in hectares:
class_en
still_in_production    2288.969268
agriculture            1739.232797
open                    582.307073
afforested               79.734218
water                    16.199157
Name: area_ha, dtype: float64


In [10]:
# production polygons only, clipped to the training region, in the raster's CRS
train_labels = peat_wgs.cx[22.4:22.8, 62.2:62.5]
train_labels = train_labels[train_labels["class_en"] == "still_in_production"]
train_labels = train_labels.to_crs("EPSG:32634")

train_labels.to_file("../data/processed/binary_model_train_labels_2021.gpkg", driver="GPKG")
print(len(train_labels), "polygons, CRS", train_labels.crs)

193 polygons, CRS EPSG:32634


# Tiling the Imagery

In [11]:
binary_train_raster_path_2021 = "../data/raw/raster/binary_model_train_2021.tif"
binary_train_vector_path_2021 = "../data/processed/binary_model_train_labels_2021.gpkg"
out_folder = "../data/training/binary_model_train_tiles"

tiles = geoai.export_geotiff_tiles(
    in_raster=binary_train_raster_path_2021,
    out_folder=out_folder,
    in_class_data=binary_train_vector_path_2021,
    tile_size=256,
    stride=128,
    buffer_radius=0,
)

'class' not found in vector data. Using default class ID 1.


Generating tiles:   0%|          | 0/416 [00:00<?, ?it/s]

In [13]:
import glob
import rasterio

In [14]:
labels = sorted(glob.glob(f"{out_folder}/labels/*.tif"))
images = sorted(glob.glob(f"{out_folder}/images/*.tif"))
print(len(images), "images,", len(labels), "labels")

maxes = []
for path in labels:
    with rasterio.open(path) as src:
        maxes.append(int(src.read(1).max()))

print("highest value in any label:", max(maxes))
print("labels above 1:", sum(m > 1 for m in maxes))

416 images, 416 labels
highest value in any label: 1
labels above 1: 0


In [15]:
counts = []
for path in labels:
    with rasterio.open(path) as src:
        counts.append(int(src.read(1).sum()))

print("tiles with something:", sum(c > 0 for c in counts))
print("total positive pixels:", sum(counts))

tiles with something: 241
total positive pixels: 959742


# Training the Model

In [16]:
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=6,
    num_classes=2,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
)

KeyboardInterrupt: 